# 06 — Recherche sémantique locale avec FAISS

**Objectif :** encoder requêtes et produits localement, puis effectuer une recherche vectorielle.

**Entrées :** produits nettoyés et validation.  
**Sorties :** index FAISS local, mapping SKU et métriques.  
**Dépendances :** notebooks 02 et 03.  
**Temps estimé :** 2 à 5 minutes au premier téléchargement du modèle.  
**Ressources :** RTX 5070 pour l'encodage, FAISS CPU sous Windows.

In [ ]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import torch
import yaml
from sentence_transformers import SentenceTransformer


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Racine du projet introuvable.")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.metrics import evaluate_rankings

CONFIG = yaml.safe_load((ROOT / "configs" / "default.yaml").read_text(encoding="utf-8"))
TOP_K = int(CONFIG["project"]["top_k"])
MODEL_NAME = CONFIG["semantic"]["model_name"]
PROCESSED_DIR = ROOT / CONFIG["paths"]["processed_data"]
ARTIFACTS_DIR = ROOT / CONFIG["paths"]["artifacts"]
REPORTS_DIR = ROOT / CONFIG["paths"]["reports"]
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

products = pd.read_csv(PROCESSED_DIR / "products_clean.csv", dtype={"sku": str}).fillna("")
validation = pd.read_csv(PROCESSED_DIR / "validation_clicks.csv", dtype={"sku": str}).fillna("")

device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 64 if device == "cuda" else 16
print(f"Modèle={MODEL_NAME}; device={device}; batch_size={batch_size}")

## Encodage des produits

In [ ]:
started = time.perf_counter()
model = SentenceTransformer(MODEL_NAME, device=device)
product_embeddings = model.encode(
    products["document"].tolist(),
    batch_size=batch_size,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False,
).astype("float32")
encoding_seconds = time.perf_counter() - started

index = faiss.IndexFlatIP(product_embeddings.shape[1])
index.add(product_embeddings)
assert index.ntotal == len(products)
print(f"{len(products)} produits encodés en {encoding_seconds:.2f}s")


def semantic_ranking(query: str, k: int = TOP_K) -> list[str]:
    query_embedding = model.encode(
        [query], normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False
    ).astype("float32")
    _, indices = index.search(query_embedding, min(k, len(products)))
    return products.iloc[indices[0]]["sku"].astype(str).tolist()


for query in ("space hero shooter", "open world cars", "pirate adventure"):
    print(query, "→", semantic_ranking(query))

## Évaluation et persistance

In [ ]:
actual_by_query = validation.groupby("query_key")["sku"].agg(lambda values: set(map(str, values)))
query_text_by_key = validation.groupby("query_key")["query_text"].first()
predictions = [semantic_ranking(query_text_by_key.loc[key]) for key in actual_by_query.index]
assert all(len(items) == len(set(items)) == TOP_K for items in predictions)

metrics = evaluate_rankings(actual_by_query.tolist(), predictions, TOP_K)
report = {
    "model": MODEL_NAME,
    "device": device,
    "embedding_dimension": int(product_embeddings.shape[1]),
    "product_count": int(len(products)),
    "encoding_seconds": encoding_seconds,
    "faiss_backend": "cpu-index-flat-ip",
    **metrics,
}

faiss.write_index(index, str(ARTIFACTS_DIR / "semantic.index"))
(ARTIFACTS_DIR / "semantic_skus.json").write_text(
    json.dumps(products["sku"].astype(str).tolist()), encoding="utf-8"
)
(REPORTS_DIR / "metrics_semantic.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(json.dumps(report, indent=2, ensure_ascii=False))

## Conclusion

L'index FAISS reste sur CPU pour une installation Windows simple. Le coût dominant — l'encodage
Transformer — utilise bien CUDA sur la RTX 5070 lorsque disponible.